# Extra security prompts (V10 to V39, written by member C): base vs fine-tuned, bf16 (adapter merged)
15 vulnerable + 15 safe snippets (each safe snippet is the fixed twin of a vulnerable one). Same generation code as every other run.
Inputs: the adapter (Kaggle Input) and `prompts_securite_extra.json` (found in the cloned repo, or add it as an Input file).
These prompts are an EVALUATION set only: never use them for training.
Output: one CSV per model with the answers, to be graded blind with member C's label rules for this set.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "main"

if not os.path.exists("../config.json"):
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 117 (delta 33), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 390.14 KiB | 3.86 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
# torchao 0.10.0 (preinstalled on Kaggle) is rejected by peft 0.21.2 (needs >=0.16). We do not use torchao, so remove it.
!pip uninstall -y -q torchao
!pip install -q psutil nvidia-ml-py
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-823f4313-20ef-999e-53d5-078bf2040702)
GPU 1: Tesla T4 (UUID: GPU-f30765e3-29f4-5f09-533f-130bf5bd5c11)


In [3]:
import json, time, sys, re, threading
from math import comb
import torch, psutil, pynvml
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "qwen"
# ====================================================================

for f in ["../config.json", "../config_complement.json", "../prompts_securite.json", "../humaneval_subset.json"]:
    assert os.path.exists(f), f"{f} introuvable : ajoutez-le a la branche clonee (BRANCH) puis relancez."

config = json.load(open("../config.json"))
CC = json.load(open("../config_complement.json"))
SEC_PROMPTS = json.load(open("../prompts_securite.json", encoding="utf-8"))["prompts"]
PK, MON = CC["passk"], CC["monitor"]

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME, "| transformers", transformers.__version__)

Qwen/Qwen2.5-Coder-1.5B-Instruct | qwen2.5-coder-1.5b-instruct | Tesla T4 | transformers 5.20.2


In [4]:
torch.manual_seed(42)


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


In [5]:
import glob, zipfile, hashlib, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert config["precision"] == "bf16"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
EXPECTED_SHA = "02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3"   # adapter evaluated in the 20-problem runs

def find_adapter():
    cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not cfgs:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        assert zips, "Add adapter_run1.zip as a Kaggle Input, then rerun"
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working/adapter_run1")
        cfgs = glob.glob("/kaggle/working/adapter_run1/**/adapter_config.json", recursive=True)
    return os.path.dirname(cfgs[0])

def load(kind):
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16)
    sha = None
    if kind == "ft":
        d = find_adapter()
        sha = hashlib.sha256(open(os.path.join(d, "adapter_model.safetensors"), "rb").read()).hexdigest()
        assert sha == EXPECTED_SHA, "adapter differs from the one evaluated before"
        m = PeftModel.from_pretrained(m, d).merge_and_unload()
        assert not any("lora" in n.lower() for n, _ in m.named_parameters())
    m.eval()
    return m, sha
print("ready")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ready


In [8]:
PF = "../prompts_securite_extra.json"
if not os.path.exists(PF):
    hits = glob.glob("/kaggle/input/**/prompts_securite_extra.json", recursive=True)
    assert hits, "prompts_securite_extra.json not found: push it to the repo main branch or add it as a Kaggle Input"
    PF = hits[0]
extra = json.load(open(PF, encoding="utf-8"))["prompts"]
assert len(extra) == 30 and sum(p["has_vuln"] for p in extra) == 15, "unexpected prompt set"
print("prompts:", len(extra), "| vulnerable:", sum(p["has_vuln"] for p in extra))

def run_extra(kind):
    slug = f"{model_slug}-secextra-" + ("base" if kind == "base" else "ft-run1-bf16")
    rows = []
    for p in extra:
        gen = generate(p["prompt"]); resp = gen.pop("response")
        kw = [k.lower() for k in p.get("keywords", [])]
        rows.append({"id": p["id"], "prompt": p["prompt"], "response": resp, **gen, "has_vuln": p["has_vuln"],
                     "expected_vuln": p["expected_vuln"], "mot_cle_trouve": any(k in resp.lower() for k in kw) if kw else None,
                     "vuln_label": None})
        print(kind, p["id"], "| keyword:", rows[-1]["mot_cle_trouve"])
    pd.DataFrame(rows).to_csv(f"../results/securite_extra_{slug}.csv", index=False)
    return slug

sha_ft = None
for kind in ("base", "ft"):
    model, sha = load(kind)
    if sha: sha_ft = sha
    warm = generate("Write a Python function add(a, b) that returns a + b.")
    assert "def" in warm["response"], "empty or incoherent answer: stop and check the model"
    run_extra(kind)
    del model; gc.collect(); torch.cuda.empty_cache()

prompts: 30 | vulnerable: 15


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

base V10 | keyword: True
base V11 | keyword: False
base V12 | keyword: True
base V13 | keyword: True
base V14 | keyword: True
base V15 | keyword: True
base V16 | keyword: True
base V17 | keyword: True
base V18 | keyword: True
base V19 | keyword: True
base V20 | keyword: False
base V21 | keyword: True
base V22 | keyword: True
base V23 | keyword: True
base V24 | keyword: True
base V25 | keyword: None
base V26 | keyword: None
base V27 | keyword: None
base V28 | keyword: None
base V29 | keyword: None
base V30 | keyword: None
base V31 | keyword: None
base V32 | keyword: None
base V33 | keyword: None
base V34 | keyword: None
base V35 | keyword: None
base V36 | keyword: None
base V37 | keyword: None
base V38 | keyword: None
base V39 | keyword: None


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ft V10 | keyword: True
ft V11 | keyword: False
ft V12 | keyword: True
ft V13 | keyword: True
ft V14 | keyword: True
ft V15 | keyword: False
ft V16 | keyword: True
ft V17 | keyword: True
ft V18 | keyword: True
ft V19 | keyword: False
ft V20 | keyword: False
ft V21 | keyword: True
ft V22 | keyword: True
ft V23 | keyword: True
ft V24 | keyword: True
ft V25 | keyword: None
ft V26 | keyword: None
ft V27 | keyword: None
ft V28 | keyword: None
ft V29 | keyword: None
ft V30 | keyword: None
ft V31 | keyword: None
ft V32 | keyword: None
ft V33 | keyword: None
ft V34 | keyword: None
ft V35 | keyword: None
ft V36 | keyword: None
ft V37 | keyword: None
ft V38 | keyword: None
ft V39 | keyword: None


In [ ]:
import platform, datetime, shutil
from importlib.metadata import version
meta = {"experiment": "extra security prompts V10-V39, base vs fine-tuned, bf16", "gpu": GPU_NAME, "adapter_sha256": sha_ft,
        "prompt_file": PF, "generation_params": GEN_PARAMS,
        "versions": {k: version(k) for k in ["torch", "transformers", "peft", "accelerate", "datasets"]},
        "python": platform.python_version(), "date_utc": datetime.datetime.utcnow().isoformat(),
        "note": "torchao uninstalled (preinstalled 0.10.0 incompatible with peft 0.21.2)"}
json.dump(meta, open("../results/run_meta_secextra.json", "w"), indent=2)
os.makedirs("/kaggle/working/secextra", exist_ok=True)
for f in glob.glob("../results/securite_extra_*.csv") + ["../results/run_meta_secextra.json"]:
    shutil.copy(f, "/kaggle/working/secextra/")
shutil.make_archive("/kaggle/working/secextra_results", "zip", "/kaggle/working/secextra")
print(json.dumps(meta, indent=2)); print("\nTo download: /kaggle/working/secextra_results.zip")